In [ ]:
!pip install datasets

In [ ]:
!nvidia-smi

In [ ]:
"""
🎯 E8-Transformer Lila-E8
This project is licensed under the GNU Affero General Public License v3.0 or later (AGPL-3.0-or-later).
Commercial Licensing: For proprietary R&D, integration into private AI stacks, or hardware implementation,
please contact the Architect directly.
Copyright (C) 2026 Anatolii Kornienko This program is free software: you can redistribute it and/or modify
it under the terms of the GNU Affero General Public License
as published by the Free Software Foundation, either version 3 of the License, or any later version.

This program is free software: you can redistribute it and/or modify
it under the terms of the GNU Affero General Public License as published by
the Free Software Foundation, either version 3 of the License, or
(at your option) any later version.

This program is distributed in the hope that it will be useful,
but WITHOUT ANY WARRANTY; without even the implied warranty of
MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.  See the
GNU Affero General Public License for more details.

You should have received a copy of the GNU Affero General Public License
along with this program.  If not, see <https://www.gnu.org/licenses/agpl-3.0.txt/>.
"""

import os
import requests
import sentencepiece as spm

model_path = 'e8_morpheme.model'

if not os.path.exists(model_path):
    print("🔧 Tokenizer not found. Training on Shakespeare...")
    # Скачиваем Shakespeare
    url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
    text = requests.get(url).text
    temp_file = "input_text.txt"
    with open(temp_file, "w", encoding="utf-8") as f:
        f.write(text)
    # Обучаем BPE (vocab_size=2048, как у вас было)
    spm.SentencePieceTrainer.train(
        input=temp_file,
        model_prefix='e8_morpheme',
        vocab_size=2048,
        model_type='bpe',
        character_coverage=1.0,
        byte_fallback=True,
        unk_id=0, pad_id=1, bos_id=2, eos_id=3
    )
    print("✅ The tokenizer is trained and saved as e8_morpheme.model")
else:
    print("✅ The tokenizer already exists, loading...")

# Теперь загружаем
sp = spm.SentencePieceProcessor(model_file='e8_morpheme.model')
vocab_size = sp.get_piece_size()
print(f"💎 Vocab Size: {vocab_size}")

In [ ]:
"""
🎯 E8-Transformer Lila-E8
This project is licensed under the GNU Affero General Public License v3.0 or later (AGPL-3.0-or-later).
Commercial Licensing: For proprietary R&D, integration into private AI stacks, or hardware implementation,
please contact the Architect directly.
Copyright (C) 2026 Anatolii Kornienko This program is free software: you can redistribute it and/or modify
it under the terms of the GNU Affero General Public License
as published by the Free Software Foundation, either version 3 of the License, or any later version.

This program is free software: you can redistribute it and/or modify
it under the terms of the GNU Affero General Public License as published by
the Free Software Foundation, either version 3 of the License, or
(at your option) any later version.

This program is distributed in the hope that it will be useful,
but WITHOUT ANY WARRANTY; without even the implied warranty of
MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.  See the
GNU Affero General Public License for more details.

You should have received a copy of the GNU Affero General Public License
along with this program.  If not, see <https://www.gnu.org/licenses/agpl-3.0.txt/>.
"""

import gc
import torch
import torch.nn as nn
import torch.nn.functional as F
import requests
import os
import re
import math
from dataclasses import dataclass
from typing import Optional
import time
from torch import Tensor
import sentencepiece as spm
import io
from datasets import load_dataset
import random
from collections import deque

# 0. GPU AUTO-DETECTION
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"📡 Device: {device} | {'🔥 GPU ACTIVE' if device == 'cuda' else '⚠️ CPU MODE'}")

# 1. LOADING A READY-MADE BPE TOKENIZER (trained on Shakespeare)
# The e8_morpheme.model file should be located in the Colab working folder
sp = spm.SentencePieceProcessor(model_file='e8_morpheme.model')
vocab_size = sp.get_piece_size()
print(f"💎 Tokenizer loaded. Vocab Size: {vocab_size}")

# 2. ENCODING/DECODING FUNCTIONS (without out_type – compatible with all versions)
def encode(text):
    return sp.encode(text)

def decode(tokens):
    return sp.decode(tokens)

# 3. CONNECT TO THE ENDLESS STREAM OF TINYSTORIES
dataset = load_dataset("roneneldan/TinyStories", streaming=True, split="train")
train_iter = iter(dataset)
print("🌊 TinyStories streaming activated")


def get_batch_streaming(iterator, batch_size, block_size, device, pad_token_id=1, buffer_size=200):
    x_batch, y_batch = [], []
    buffer = deque()


    while len(buffer) < buffer_size:
        try:
            ex = next(iterator)
            buffer.append(ex)
        except StopIteration:
            break

    while len(x_batch) < batch_size:
        if not buffer:
            return None, None
        ex = random.choice(buffer)
        tokens = encode(ex['text'])
        if len(tokens) <= 1:
            continue


        if len(tokens) > block_size + 1:
            start = random.randint(0, len(tokens) - block_size - 1)
            chunk = tokens[start:start + block_size + 1]
        else:

            pad_len = block_size + 1 - len(tokens)
            chunk = tokens + [pad_token_id] * pad_len

        x_batch.append(chunk[:-1])
        y_batch.append(chunk[1:])

    try:
        new_ex = next(iterator)
        buffer.append(new_ex)
        buffer.popleft()
    except StopIteration:
        pass

    X = torch.tensor(x_batch, dtype=torch.long, device=device)
    Y = torch.tensor(y_batch, dtype=torch.long, device=device)
    return X, Y

# 5. BATCH TEST (optional)
xb, yb = get_batch_streaming(train_iter, batch_size=4, block_size=256, device=device)
print(f"✅ Trial batch: X {xb.shape}, Y {yb.shape} | tokens: {xb.numel()}")

In [ ]:
"""
🎯 E8-Transformer Lila-E8
This project is licensed under the GNU Affero General Public License v3.0 or later (AGPL-3.0-or-later).
Commercial Licensing: For proprietary R&D, integration into private AI stacks, or hardware implementation,
please contact the Architect directly.
Copyright (C) 2026 Anatolii Kornienko This program is free software: you can redistribute it and/or modify
it under the terms of the GNU Affero General Public License
as published by the Free Software Foundation, either version 3 of the License, or any later version.

This program is free software: you can redistribute it and/or modify
it under the terms of the GNU Affero General Public License as published by
the Free Software Foundation, either version 3 of the License, or
(at your option) any later version.

This program is distributed in the hope that it will be useful,
but WITHOUT ANY WARRANTY; without even the implied warranty of
MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.  See the
GNU Affero General Public License for more details.

You should have received a copy of the GNU Affero General Public License
along with this program.  If not, see <https://www.gnu.org/licenses/agpl-3.0.txt/>.
"""


# ==================== CONFIGURATION ====================
from dataclasses import dataclass
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor

@dataclass
class E8Config:
    vocab_size: int
    d_model: int = 512
    n_layers: int = 12
    n_heads: int = 8
    block_size: int = 512
    dropout: float = 0.05
    e8_scale: float = 0.0266
    bias: bool = False
    e8_strength: float = 0.01
    temp: float = 0.1
    tie_weights: bool = True


# ============================================================
# MATHEMATICALLY CORRECT ROOTS OF THE E8 SYSTEM (240 PIECES)
# ============================================================

@torch.jit.script
def _generate_e8_type1_roots() -> Tensor:
    roots = torch.zeros(112, 8, dtype=torch.float32)
    idx = 0
    for i in range(8):
        for j in range(i + 1, 8):
            for sign_i in [-1.0, 1.0]:
                for sign_j in [-1.0, 1.0]:
                    roots[idx, i] = sign_i
                    roots[idx, j] = sign_j
                    idx += 1
    return roots

@torch.jit.script
def _generate_e8_type2_roots() -> Tensor:
    roots = torch.zeros(128, 8, dtype=torch.float32)
    idx = 0
    for mask in range(256):
        negative_count = 0
        root = torch.zeros(8, dtype=torch.float32)
        for bit in range(8):
            if (mask >> bit) & 1:
                root[bit] = -0.5
                negative_count += 1
            else:
                root[bit] = 0.5
        if negative_count % 2 == 0:
            roots[idx] = root
            idx += 1
    return roots

def _verify_e8_mathematical_properties(roots: Tensor) -> None:
    assert roots.shape == (240, 8), f"Incorrect dimension: {roots.shape}"
    norms = torch.norm(roots, dim=1)
    expected_norm = math.sqrt(2.0)
    max_error = torch.abs(norms - expected_norm).max().item()
    assert max_error < 1e-6, f"Max. error: {max_error}"
    unique_roots = torch.unique(roots, dim=0)
    assert unique_roots.shape[0] == 240, f"Duplicated roots! Unique: {unique_roots.shape[0]}"
    center_deviation = torch.norm(roots.sum(dim=0)).item()
    assert center_deviation < 1e-4, f"Not centered: {center_deviation}"

def _compute_production_e8_roots() -> Tensor:
    print("🧮 Generating production-ready E8 system roots...")
    type1_roots = _generate_e8_type1_roots()
    type2_roots = _generate_e8_type2_roots()
    all_roots = torch.cat([type1_roots, type2_roots], dim=0)
    _verify_e8_mathematical_properties(all_roots)
    print("✅ Production-ready E8 roots generated and verified!")
    return all_roots

_E8_ROOTS_CACHE = _compute_production_e8_roots()

# ============================================================
# JIT-COMPILED CORE OPERATIONS
# ============================================================

@torch.jit.script
def jit_e8_distance_matrix(z: Tensor, roots: Tensor) -> Tensor:
    z_norms_sq = (z * z).sum(dim=-1, keepdim=True)
    cross_terms = torch.matmul(z, roots.T)
    root_norms_sq = (roots * roots).sum(dim=-1)
    distances_sq = z_norms_sq - 2.0 * cross_terms + root_norms_sq
    return distances_sq

@torch.jit.script
def jit_soft_e8_quantization(distances_sq: Tensor, temp: float, roots: Tensor) -> Tensor:
    weights = torch.softmax(-distances_sq / temp, dim=-1)
    return torch.matmul(weights, roots)

@torch.jit.script
def jit_e8_attention_bias(q: Tensor, k: Tensor, head_dirs: Tensor, head_scales: Tensor) -> Tensor:
    B, H, T, D = q.shape
    if D >= 8:
        q_e8 = q[:, :, :, :8]
        k_e8 = k[:, :, :, :8]
        q_proj = torch.einsum('bhtd,hd->bht', q_e8, head_dirs)
        k_proj = torch.einsum('bhtd,hd->bht', k_e8, head_dirs)
        geo_bias = q_proj.unsqueeze(-1) * k_proj.unsqueeze(-2)
        return head_scales.view(1, -1, 1, 1) * geo_bias
    else:
        return torch.zeros(B, H, T, T, device=q.device, dtype=q.dtype)

# ==================== E8 ROOTS ====================
print("\n🌌 Generating E8 roots...")
roots_t1 = _generate_e8_type1_roots()
roots_t2 = _generate_e8_type2_roots()
e8_roots = torch.cat([roots_t1, roots_t2], dim=0).to(torch.float32)
e8_roots = torch.nn.functional.normalize(e8_roots, p=2, dim=1)
print(f"✅ 240 E8 roots created")
print(f"   Dimension: {e8_roots.shape}")
print(f"   Norm: {torch.norm(e8_roots[0]):.4f} (should be √2 ≈ 1.4142)")

# ==================== E8 QUANTIZER ====================
class E8Quantizer(nn.Module):
    def __init__(self, temp=0.1):
        super().__init__()
        self.temp = temp
        self.register_buffer('roots', e8_roots)
        self.register_buffer('roots_norm_sq', (e8_roots ** 2).sum(dim=1))

    def forward(self, x):
        roots = self.roots
        roots_norm_sq = self.roots_norm_sq
        x_norm_sq = (x ** 2).sum(dim=-1, keepdim=True)
        cross = x @ roots.T
        distances = x_norm_sq - 2 * cross + roots_norm_sq
        weights = F.softmax(-distances / self.temp, dim=-1)
        quantized = weights @ roots
        return x + (quantized - x).detach()

    def hard_quantize(self, x):
        distances = torch.cdist(x, self.roots)
        indices = torch.argmin(distances, dim=-1)
        return self.roots[indices]

# ==================== E8 LINEAR ====================
class E8Linear(nn.Module):
    def __init__(self, in_features: int, cfg: E8Config):
        super().__init__()
        self.cfg = cfg
        self.in_features = in_features
        if cfg.tie_weights:
            self.weight = nn.Parameter(torch.randn(in_features, 8) * 0.02)
            self.proj = lambda x: F.linear(x, self.weight.t(), None)
            self.unproj = lambda x: F.linear(x, self.weight, None)
        else:
            self.proj = nn.Linear(in_features, 8, bias=cfg.bias)
            self.unproj = nn.Linear(8, in_features, bias=cfg.bias)
        self.quantizer = E8Quantizer(cfg.temp)
        self.alpha = nn.Parameter(torch.tensor(cfg.e8_strength))

    def forward(self, x):
        z = self.proj(x)
        z_quantized = self.quantizer(z)
        z_mixed = z + self.alpha * (z_quantized - z)
        return self.unproj(z_mixed)


class E8GraphResonator(nn.Module):
    def __init__(self, d_model=128):
        super().__init__()
        self.d_model = d_model
        self.register_buffer("e8_roots", self._get_e8_basis(d_model))
        self.reasoning_gate = nn.Parameter(torch.randn(d_model, d_model) * 0.02)

    def _get_e8_basis(self, d_model):
        raw_roots = self._generate_240_roots() # (240, 8)
        raw_roots = torch.nn.functional.normalize(raw_roots, p=2, dim=1)
        proj = torch.randn(8, d_model) / (8**0.5)

        return torch.matmul(raw_roots, proj) # (240, d_model)

    def _generate_240_roots(self):
        roots = []
        # Тип 1: (±1, ±1, 0^6)
        for i in range(8):
            for j in range(i + 1, 8):
                for s1 in [-1.0, 1.0]:
                    for s2 in [-1.0, 1.0]:
                        r = [0.0]*8
                        r[i], r[j] = s1, s2
                        roots.append(r)
        # Тип 2: (±0.5)^8 (четное кол-во минусов)
        for i in range(256):
            r = [0.5 if not ((i >> b) & 1) else -0.5 for b in range(8)]
            if sum(1 for x in r if x < 0) % 2 == 0:
                roots.append(r)
        return torch.tensor(roots, dtype=torch.float32)


    def encode_relation(self, node_a_idx, node_b_idx):
        root_a = self.e8_roots[node_a_idx % 240]
        root_b = self.e8_roots[node_b_idx % 240]
        relation = torch.outer(root_a, root_b)
        with torch.no_grad():
            self.reasoning_gate.add_(relation * 1) #(relation * 1.0) (instead of 0.01). We need a strong "fingerprint".

    def forward(self, query_node_idx):
        """
        Graph search: Submit a node, get resonance with related nodes
        """
        query_vec = self.e8_roots[query_node_idx % 240].unsqueeze(0)
        latent_response = torch.matmul(query_vec, self.reasoning_gate)
        scores = torch.matmul(F.normalize(latent_response, dim=-1),
                              F.normalize(self.e8_roots, dim=-1).t())

        return scores #

# ==================== E8 ATTENTION ====================
class E8Attention(nn.Module):
    def __init__(self, cfg: E8Config):
        super().__init__()
        assert cfg.d_model % cfg.n_heads == 0
        self.n_heads = cfg.n_heads
        self.head_dim = cfg.d_model // cfg.n_heads
        self.scale = 1.0 / math.sqrt(self.head_dim)
        self.qkv = nn.Linear(cfg.d_model, 3 * cfg.d_model, bias=cfg.bias)
        self.out = nn.Linear(cfg.d_model, cfg.d_model, bias=cfg.bias)
        self.dropout = nn.Dropout(cfg.dropout)
        self.register_buffer('head_dirs', e8_roots[:cfg.n_heads])
        self.head_scales = nn.Parameter(torch.zeros(cfg.n_heads))
        self.register_buffer("causal_mask", torch.tril(torch.ones(1, 1, cfg.block_size, cfg.block_size)))

    def forward(self, x):
        B, T, C = x.shape
        qkv = self.qkv(x).reshape(B, T, 3, self.n_heads, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)
        q, k, v = qkv.unbind(0)
        scores = (q @ k.transpose(-2, -1)) * self.scale
        scores = scores.masked_fill(self.causal_mask[:,:,:T,:T] == 0, float('-inf'))
        if self.head_dim >= 8:
            q_e8 = q[..., :8]
            k_e8 = k[..., :8]
            q_proj = torch.einsum('bhtd,hd->bht', q_e8, self.head_dirs)
            k_proj = torch.einsum('bhtd,hd->bht', k_e8, self.head_dirs)
            geo_bias = q_proj.unsqueeze(-1) * k_proj.unsqueeze(-2)
            scores = scores + self.head_scales.view(1, -1, 1, 1) * geo_bias
        attn = F.softmax(scores, dim=-1)
        attn = self.dropout(attn)

        out = (attn @ v).transpose(1, 2).reshape(B, T, C)
        self.attn_weights = attn  #self.attn_weights = attn #добавил для теста
        return self.out(out)

# ==================== E8 TRANSFORMER LAYER ====================
class E8TransformerLayer(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.ln_attn = nn.LayerNorm(cfg.d_model)
        self.attn = E8Attention(cfg)
        self.ln_res = nn.LayerNorm(cfg.d_model)
        self.to_e8 = nn.Linear(cfg.d_model, 8, bias=False)
        self.from_e8 = nn.Linear(8, cfg.d_model, bias=False)
        self.quantizer = E8Quantizer(temp=cfg.temp)
        self.e8_scale = nn.Parameter(torch.tensor([cfg.e8_strength]))
        self.ln_ffn = nn.LayerNorm(cfg.d_model)
        self.ffn = nn.Sequential(
            nn.Linear(cfg.d_model, 4 * cfg.d_model),
            nn.GELU(),
            nn.Linear(4 * cfg.d_model, cfg.d_model),
            nn.Dropout(cfg.dropout)
        )

    def forward(self, x):
        x = x + self.attn(self.ln_attn(x))
        res_in = self.ln_res(x)
        e8_space = self.to_e8(res_in)
        res_feat = self.quantizer(e8_space)
        if self.training:
            res_feat = res_feat * (1.0 + 0.001 * torch.randn_like(res_feat))
        res_out = self.from_e8(res_feat)
        if self.training:
            res_out = res_out + torch.randn_like(res_out) * 0.005
        x = x + self.e8_scale * res_out
        x = x + self.ffn(self.ln_ffn(x))
        return x

# ==================== E8 TRANSFORMER (CORE, NO HEAD) ====================
class E8Transformer(nn.Module):
    def __init__(self, cfg: E8Config):
        super().__init__()
        self.cfg = cfg
        self.layers = nn.ModuleList([E8TransformerLayer(cfg) for _ in range(cfg.n_layers)])
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.pos_emb = nn.Parameter(torch.zeros(1, cfg.block_size, cfg.d_model))
        self.final_norm = nn.LayerNorm(cfg.d_model)
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.trunc_normal_(module.weight, std=0.04)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.LayerNorm):
            if module.bias is not None:
                nn.init.zeros_(module.bias)
            if module.weight is not None:
                nn.init.ones_(module.weight)

    def forward(self, idx, targets=None):
        if idx.dim() == 3:
            b, t, _ = idx.size()
            x = idx
        else:
            b, t = idx.size()
            x = self.tok_emb(idx) + self.pos_emb[:, :t, :]
        for layer in self.layers:
            x = layer(x)
        x = self.final_norm(x)
        return x

# ==================== E8-GPT  ====================
class E8GPT(nn.Module):
    def __init__(self, cfg: E8Config):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.pos_emb = nn.Parameter(torch.zeros(1, cfg.block_size, cfg.d_model))
        self.core = E8Transformer(cfg)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.trunc_normal_(module.weight, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.trunc_normal_(module.weight, std=0.02)

    def forward(self, idx, targets=None):
        if idx.dim() > 2:
            idx = idx.squeeze()
        b, t = idx.size()
        x = self.tok_emb(idx) + self.pos_emb[:, :t, :]
        hidden_states = self.core(x)
        logits = self.head(hidden_states)
        loss = None
        if targets is not None:
            if targets.dim() > 2:
                targets = targets.squeeze()
            logits_flat = logits.view(-1, logits.size(-1))
            targets_flat = targets.view(-1)
            loss = F.cross_entropy(logits_flat, targets_flat)
        return logits, loss

    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        for _ in range(max_new_tokens):
            idx_cond = idx if idx.size(1) <= self.cfg.block_size else idx[:, -self.cfg.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('Inf')
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

# ====================================================================
# CREATING THE MODEL (ONLY AFTER vocab_size IS DEFINED)
# ========================================================================
# Make sure the vocab_size variable is defined (tokenizer loaded)
# Example: vocab_size = 2048 (or from sp.get_piece_size())
# The following creates a model instance. This code should be executed AFTER the tokenizer is loaded.

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"📡 Device: {device}")

# It is assumed that vocab_size is already defined (for example, from the loaded SentencePieceProcessor)
# If not, set it temporarily: vocab_size = 2048

# Create the config and model
cfg = E8Config(vocab_size=vocab_size)
model = E8GPT(cfg).to(device)

print(f"💎 Модель E8-Sovereign создана.")
print(f"📦 Параметров: {sum(p.numel() for p in model.parameters())/1e6:.2f}M")
print(f"   Архитектура: d_model={cfg.d_model}, layers={cfg.n_layers}, heads={cfg.n_heads}")

In [ ]:
"""
🎯 E8-Transformer Lila-E8
This project is licensed under the GNU Affero General Public License v3.0 or later (AGPL-3.0-or-later).
Commercial Licensing: For proprietary R&D, integration into private AI stacks, or hardware implementation,
please contact the Architect directly.
Copyright (C) 2026 Anatolii Kornienko This program is free software: you can redistribute it and/or modify
it under the terms of the GNU Affero General Public License
as published by the Free Software Foundation, either version 3 of the License, or any later version.

This program is free software: you can redistribute it and/or modify
it under the terms of the GNU Affero General Public License as published by
the Free Software Foundation, either version 3 of the License, or
(at your option) any later version.

This program is distributed in the hope that it will be useful,
but WITHOUT ANY WARRANTY; without even the implied warranty of
MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.  See the
GNU Affero General Public License for more details.

You should have received a copy of the GNU Affero General Public License
along with this program.  If not, see <https://www.gnu.org/licenses/agpl-3.0.txt/>.
"""

# ========================
# 1. GOOGLE DRIVE (checkpoints)
# ========================
from google.colab import drive
drive.mount('/content/drive')

checkpoint_dir = '/content/drive/MyDrive/e8_tinystories_checkpoints3'
os.makedirs(checkpoint_dir, exist_ok=True)

def save_checkpoint(step, model, optimizer, loss, is_best=False):
    """Saves the checkpoint to Google Drive."""
    filename = 'best_model.pt' if is_best else f'checkpoint_step_{step}.pt'
    path = os.path.join(checkpoint_dir, filename)
    torch.save({
        'step': step,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        #'scheduler_state_dict': scheduler.state_dict(),
        'loss': loss,
        'config': cfg,  #
    }, path)
    print(f'💾 Checkpoint saved: {path} (loss={loss:.4f})')

def load_latest_checkpoint(model, optimizer):
    files = [f for f in os.listdir(checkpoint_dir) if f.startswith('checkpoint_step_')]
    if not files:
        # попробуем загрузить best_model.pt
        best_path = os.path.join(checkpoint_dir, 'best_model.pt')
        if os.path.exists(best_path):
            checkpoint = torch.load(best_path, map_location=device, weights_only=False)  # <---  weights_only=False
            model.load_state_dict(checkpoint['model_state_dict'])
            optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
            #if 'scheduler_state_dict' in checkpoint:
              #scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
            print(f'🔄 Loading best_model.pt, step {checkpoint["step"]}, loss {checkpoint["loss"]:.4f}')
            return checkpoint['step']
        print('🆕 There are no checkpoints, start from scratch.')
        return 0
    steps = [int(f.split('_')[-1].split('.')[0]) for f in files]
    latest_step = max(steps)

    latest_file = f'checkpoint_step_{latest_step}.pt'

    path = os.path.join(checkpoint_dir, latest_file)
    checkpoint = torch.load(path, map_location=device, weights_only=False)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    print(f'🔄 Checkpoint loaded: step {latest_step}, loss {checkpoint["loss"]:.4f}')
    return latest_step

In [ ]:
"""
🎯 E8-Transformer Lila-E8
This project is licensed under the GNU Affero General Public License v3.0 or later (AGPL-3.0-or-later).
Commercial Licensing: For proprietary R&D, integration into private AI stacks, or hardware implementation,
please contact the Architect directly.
Copyright (C) 2026 Anatolii Kornienko This program is free software: you can redistribute it and/or modify
it under the terms of the GNU Affero General Public License
as published by the Free Software Foundation, either version 3 of the License, or any later version.

This program is free software: you can redistribute it and/or modify
it under the terms of the GNU Affero General Public License as published by
the Free Software Foundation, either version 3 of the License, or
(at your option) any later version.

This program is distributed in the hope that it will be useful,
but WITHOUT ANY WARRANTY; without even the implied warranty of
MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.  See the
GNU Affero General Public License for more details.

You should have received a copy of the GNU Affero General Public License
along with this program.  If not, see <https://www.gnu.org/licenses/agpl-3.0.txt/>.
"""

from google.colab import userdata
userdata.get('HF_TOKEN') #you need HF_TOKEN

# ========================
# 2. STREAMS
# ========================
# Train stream
train_dataset = load_dataset("roneneldan/TinyStories", streaming=True, split="train")
train_iter = iter(train_dataset)

# Validation stream (отдельный сплит)
val_dataset = load_dataset("roneneldan/TinyStories", streaming=True, split="validation")
val_iter = iter(val_dataset)

print("🌊  TinyStories are ready (train + validation)")

# ========================
# 3. PARAMS
# ========================
batch_size = 4         # Can be reduced to 16 if there is not enough memory
block_size = cfg.block_size
learning_rate = 5e-5 # 1e-5 #1e-4 #3e-4
total_steps = 150000    # How many steps
save_every = 1000       # Save a checkpoint every 1000 steps
log_every = 200         # Print loss every 100 steps
gen_every = 1000        # Generate text every 1000 steps

# ========================
# 4. CHECKPOINT OPTIMIZATION
# ========================
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
start_step = load_latest_checkpoint(model, optimizer)
#scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10000, eta_min=1e-5)


# ========================
# 5. TRAINING
# ========================
print(f"\n🚀 Start training from step {start_step} to {total_steps}")
model.train()

for _ in range(5): next(train_iter)
print("💎 The stream is warmed up. Entering Resonance...")

for step in range(start_step + 1, total_steps + 1):

    xb, yb = get_batch_streaming(train_iter, batch_size, block_size, device)
    if xb is None:
        train_iter = iter(train_dataset)
        continue


    logits, loss = model(xb, yb)


    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()


    best_val_loss = float('inf')
    val_loss = float('inf')


    # --- LOG ---
    if step % log_every == 0:
        print(f"📊 Step {step:5d} | Train Loss: {loss.item():.4f}")

    # --- Validation (every log_every steps) ---
    if step % log_every == 0:
        model.eval()
        with torch.no_grad():
            xb_val, yb_val = get_batch_streaming(val_iter, batch_size, block_size, device)
            if xb_val is None:
                val_iter = iter(val_dataset)
                xb_val, yb_val = get_batch_streaming(val_iter, batch_size, block_size, device)
            if xb_val is not None:
                _, val_loss = model(xb_val, yb_val)
                print(f"         | Validation Loss: {val_loss.item():.4f}")
        model.train()


    # --- Example generation (every gen_every steps) ---
    if step % gen_every == 0:
        model.eval()
        with torch.no_grad():
            test_prompt = "who is Lila? "
            context = torch.tensor([encode(test_prompt)], dtype=torch.long, device=device)
            print(f"\n🎭 Generation (step {step}): ", end='')
            for _ in range(50):
                idx_cond = context[:, -block_size:]
                logits_gen, _ = model(idx_cond)
                logits_gen = logits_gen[0, -1, :].clone()

                # Штраф за повтор недавних токенов
                past_tokens = context[0, -20:].tolist()
                for t in set(past_tokens):
                    count = past_tokens.count(t)
                    logits_gen[t] -= (2.0 + count * 1.5)

                # Запрет на повтор знака препинания
                last_token = context[0, -1].item()
                last_piece = sp.id_to_piece(last_token)
                if last_piece in ":,.!?;":
                    for p in ":,.!?;":
                        pid = sp.piece_to_id(p)
                        if pid != -1:
                            logits_gen[pid] -= 50.0

                probs = torch.softmax(logits_gen / 0.8, dim=-1)
                next_token = torch.multinomial(probs, num_samples=1)
                context = torch.cat((context, next_token.unsqueeze(0)), dim=1)

                # === ЧИСТЫЙ ВЫВОД ===
                piece = sp.id_to_piece(next_token.item())
                # Замена специальных токенов
                if piece == '<0x22>':
                    piece = '"'
                elif piece == '<0x0A>':
                    piece = '\n'
                elif piece.startswith('▁'):
                    piece = ' ' + piece[1:]
                # Если токен начинается с '<' и похож на байтовый, просто убираем его
                elif piece.startswith('<0x') and piece.endswith('>'):
                    continue  # пропускаем
                print(piece, end='', flush=True)
            print("\n" + "-"*60)
        model.train()

    # --- SAVING CHECKPOINT ---
    if step % save_every == 0:
        save_checkpoint(step, model, optimizer, loss.item())
        # We also save the best model (by validation loss)
        if 'best_val_loss' not in locals():
            best_val_loss = float('inf')
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            save_checkpoint(step, model, optimizer, val_loss.item(), is_best=True)

# --- FINAL SAVING ---
save_checkpoint(total_steps, model, optimizer, loss.item())
print("🏁 Training complete!")

In [ ]:
"""
🎯 E8-Transformer Lila-E8
This project is licensed under the GNU Affero General Public License v3.0 or later (AGPL-3.0-or-later).
Commercial Licensing: For proprietary R&D, integration into private AI stacks, or hardware implementation,
please contact the Architect directly.
Copyright (C) 2026 Anatolii Kornienko This program is free software: you can redistribute it and/or modify
it under the terms of the GNU Affero General Public License
as published by the Free Software Foundation, either version 3 of the License, or any later version.

This program is free software: you can redistribute it and/or modify
it under the terms of the GNU Affero General Public License as published by
the Free Software Foundation, either version 3 of the License, or
(at your option) any later version.

This program is distributed in the hope that it will be useful,
but WITHOUT ANY WARRANTY; without even the implied warranty of
MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.  See the
GNU Affero General Public License for more details.

You should have received a copy of the GNU Affero General Public License
along with this program.  If not, see <https://www.gnu.org/licenses/agpl-3.0.txt/>.
"""

def build_token_to_root(model, tokenizer, device):
    """
      [vocab_size]
    """
    model.eval()

    tok_emb = model.tok_emb.weight.detach()  # [vocab_size, d_model]
    to_e8 = model.core.layers[0].to_e8
    with torch.no_grad():
        e8_vectors = to_e8(tok_emb)  # [vocab_size, 8]

    roots = model.core.layers[0].quantizer.roots  # [240, 8]

    e8_vectors = F.normalize(e8_vectors, dim=-1)
    roots_norm = F.normalize(roots, dim=-1)

    sim = e8_vectors @ roots_norm.T  # [vocab_size, 240]
    token_to_root = torch.argmax(sim, dim=-1)  # [vocab_size]

    return token_to_root.cpu()

token_to_root = build_token_to_root(model, sp, device).to(device) # token_to_root на GPU

print(f"✅ Mapping: {token_to_root.shape}")

In [ ]:
path_good = '/content/drive/MyDrive/e8_tinystories_checkpoints3/checkpoint_step_169000.pt' #load latest checkpoint for inference only
checkpoint = torch.load(path_good, map_location=device, weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

In [ ]:
print(token_to_root[:10])         # roots
print(token_to_root.shape)        # (vocab_size,)
print(token_to_root.device)       # cpu

In [ ]:
print("\n💾 Saving the E8-GPT model...")

torch.save({
    'model_state_dict': model.state_dict(),
    'config': cfg,
    'e8_roots': e8_roots
}, 'e8_model_final.pth')

print(f"✅ The E8-GPT model is saved in 'e8_model_final.pth'")

In [ ]:
optimizer = torch.optim.AdamW(model.parameters())

In [ ]:
"""
🎯 E8-Transformer Lila-E8
This project is licensed under the GNU Affero General Public License v3.0 or later (AGPL-3.0-or-later).
Commercial Licensing: For proprietary R&D, integration into private AI stacks, or hardware implementation,
please contact the Architect directly.
Copyright (C) 2026 Anatolii Kornienko This program is free software: you can redistribute it and/or modify
it under the terms of the GNU Affero General Public License
as published by the Free Software Foundation, either version 3 of the License, or any later version.

This program is free software: you can redistribute it and/or modify
it under the terms of the GNU Affero General Public License as published by
the Free Software Foundation, either version 3 of the License, or
(at your option) any later version.

This program is distributed in the hope that it will be useful,
but WITHOUT ANY WARRANTY; without even the implied warranty of
MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.  See the
GNU Affero General Public License for more details.

You should have received a copy of the GNU Affero General Public License
along with this program.  If not, see <https://www.gnu.org/licenses/agpl-3.0.txt/>.
"""
resonator = E8GraphResonator(d_model=cfg.d_model).to(device)
#
# resonator.load_state_dict(torch.load('resonator.pth'))

In [ ]:

"""
🎯 E8-Transformer Lila-E8
This project is licensed under the GNU Affero General Public License v3.0 or later (AGPL-3.0-or-later).
Commercial Licensing: For proprietary R&D, integration into private AI stacks, or hardware implementation,
please contact the Architect directly.
Copyright (C) 2026 Anatolii Kornienko This program is free software: you can redistribute it and/or modify
it under the terms of the GNU Affero General Public License
as published by the Free Software Foundation, either version 3 of the License, or any later version.

This program is free software: you can redistribute it and/or modify
it under the terms of the GNU Affero General Public License as published by
the Free Software Foundation, either version 3 of the License, or
(at your option) any later version.

This program is distributed in the hope that it will be useful,
but WITHOUT ANY WARRANTY; without even the implied warranty of
MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.  See the
GNU Affero General Public License for more details.

You should have received a copy of the GNU Affero General Public License
along with this program.  If not, see <https://www.gnu.org/licenses/agpl-3.0.txt/>.
"""

def generate(model, start_str="who is Lily?", max_tokens=112, temperature=0.6,
             top_k=50, top_p=0.9, repetition_penalty=1.4, repetition_window=50,
             device=None, resonator=resonator, resonance_strength=0.07):
    """
    E8GraphResonator (optional)
    resonance_strength
    """
    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    block_size = model.cfg.block_size
    model.eval()
    if resonator is not None:
        resonator.eval()

    prompt_ids = sp.encode(start_str)
    if not prompt_ids:
        prompt_ids = [sp.bos_id() if sp.bos_id() != -1 else sp.unk_id()]

    context = torch.tensor([prompt_ids], dtype=torch.long, device=device)

    punct_ids = [pid for p in ':,.!?;' if (pid := sp.piece_to_id(p)) != -1]
    space_id = sp.piece_to_id('▁')
    if space_id == -1:
        space_id = sp.piece_to_id(' ')

    print(f"🎭 (BPE, vocab={sp.get_piece_size()})")
    print(f"\n{start_str}", end='', flush=True)


    if resonator is not None:

        e8_roots_proj = resonator.e8_roots  # [240, d_model]
        tok_emb = model.tok_emb.weight  # [vocab_size, d_model]
        tok_emb_norm = F.normalize(tok_emb, dim=-1)
        roots_norm = F.normalize(e8_roots_proj, dim=-1)
        sim = tok_emb_norm @ roots_norm.T  # [vocab_size, 240]
        token_to_root = sim.argmax(dim=-1)
        token_to_root = token_to_root.to(device)
    else:
        token_to_root = None

    generated_tokens = []
    prev_root_idx = None

    with torch.no_grad():
        for _ in range(max_tokens):
            idx_cond = context[:, -block_size:]
            logits, _ = model(idx_cond)
            logits = logits[0, -1, :].clone()

            if resonator is not None and token_to_root is not None:
                curr_token = context[0, -1].item()
                curr_root_idx = token_to_root[curr_token].item()

                if prev_root_idx is not None and prev_root_idx != curr_root_idx:
                    resonator.encode_relation(prev_root_idx, curr_root_idx)

                resonance = resonator(curr_root_idx)  # [240] или [1,240]
                if resonance.dim() > 1:
                    resonance = resonance.squeeze(0)


                resonance_bias = resonance[token_to_root]  # [vocab_size]
                logits += resonance_strength * resonance_bias

                prev_root_idx = curr_root_idx


            past_tokens = context[0, -repetition_window:].tolist()
            for t_idx in set(past_tokens):
                count = past_tokens.count(t_idx)
                logits[t_idx] -= repetition_penalty * count

            if len(context[0]) > 0:
                last_token = context[0, -1].item()
                if last_token in punct_ids:
                    logits[last_token] -= 50.0



            logits = logits / temperature
            probs = torch.softmax(logits, dim=-1)

            eos_id = sp.eos_id() if sp.eos_id() != -1 else sp.piece_to_id('<|endoftext|>')
            if eos_id == -1:
                eos_id = None

            with torch.no_grad():
                for _ in range(max_tokens):

                    next_token = torch.multinomial(probs, num_samples=1)
                    token_id = next_token.item()

                    if eos_id is not None and token_id == eos_id:
                        print()
                        break

            # Top-K
            if top_k is not None and top_k > 0:
                v, _ = torch.topk(probs, min(top_k, probs.size(-1)))
                probs[probs < v[-1]] = 0.0

            # Top-P
            if top_p is not None and 0.0 < top_p < 1.0:
                sorted_probs, sorted_indices = torch.sort(probs, descending=True)
                cumsum_probs = torch.cumsum(sorted_probs, dim=-1)
                mask = cumsum_probs > top_p
                mask[..., 1:] = mask[..., :-1].clone()
                mask[..., 0] = 0
                indices_to_remove = sorted_indices[mask]
                probs[indices_to_remove] = 0.0

            probs_sum = probs.sum()
            if probs_sum > 0:
                probs /= probs_sum
            else:
                probs = torch.zeros_like(probs)
                probs[space_id if space_id != -1 else sp.unk_id()] = 1.0

            next_token = torch.multinomial(probs, num_samples=1)
            context = torch.cat((context, next_token.unsqueeze(0)), dim=1)
            token_id = next_token.item()
            token_str = sp.id_to_piece(token_id)

            generated_tokens.append(token_id)

            # Вывод с фильтрацией мусора
            if token_str == '<pad>' or (token_str.startswith('<0x') and token_str.endswith('>')):
                continue
            if token_str == '<0x22>':
                token_str = '"'
            elif token_str == '<0x0A>':
                token_str = '\n'
            elif token_str.startswith('▁'):
                token_str = ' ' + token_str[1:]
            print(token_str, end='', flush=True)

    print()
    full_text = sp.decode(generated_tokens)
    import re
    clean_text = re.split(r'<pad>|<0x[0-9A-Fa-f]+>', full_text)[0]
    return clean_text

print("\n🎭 GENERATION (HUMAN-LIKE MODE):")
test_prompt = "Once upon a time "
generate(model, resonator=resonator, start_str=test_prompt)

In [ ]:
from google.colab import files
files.download('e8_model_final.pth')

In [ ]:
"""
🎯 E8-Transformer Lila-E8
This project is licensed under the GNU Affero General Public License v3.0 or later (AGPL-3.0-or-later).
Commercial Licensing: For proprietary R&D, integration into private AI stacks, or hardware implementation,
please contact the Architect directly.
Copyright (C) 2026 Anatolii Kornienko This program is free software: you can redistribute it and/or modify
it under the terms of the GNU Affero General Public License
as published by the Free Software Foundation, either version 3 of the License, or any later version.

This program is free software: you can redistribute it and/or modify
it under the terms of the GNU Affero General Public License as published by
the Free Software Foundation, either version 3 of the License, or
(at your option) any later version.

This program is distributed in the hope that it will be useful,
but WITHOUT ANY WARRANTY; without even the implied warranty of
MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.  See the
GNU Affero General Public License for more details.

You should have received a copy of the GNU Affero General Public License
along with this program.  If not, see <https://www.gnu.org/licenses/agpl-3.0.txt/>.
"""

# Test 1
import re
import sys
import numpy as np
import matplotlib.pyplot as plt
from typing import List, Tuple, Optional
for name, param in model.named_parameters():
    if 'head_scales' in name:
        print(name, param.data.cpu().numpy())
        # Можно также построить гистограмму
        import matplotlib.pyplot as plt
        plt.hist(param.data.cpu().numpy().flatten(), bins=20)
        plt.title(name)
        plt.show()

# Sovereign-Lila-E8: - Geometric Transformer GPT  |  

(https://github.com/SPUTNIKAI/sovereign-lila-e8 [Github](https://github.com/SPUTNIKAI/sovereign-lila-e8))

___


[![DOI](https://zenodo.org/badge/DOI/10.5281/zenodo.18729722.svg)](https://doi.org/10.5281/zenodo.18729722)
[![DOI](https://zenodo.org/badge/DOI/10.5281/zenodo.18731390.svg)](https://doi.org/10.5281/zenodo.18731390)

We introduce Sovereign-Lila-E8 (Lie Lattice Attention Language Model), a Transformer architecture that incorporates the root system of the exceptional Lie algebra
E8 into the attention mechanism. By softly quantizing hidden states into the 240 roots of E8 and adding geometric biases to attention scores, the model achieves dense semantic packing and improved long-context coherence. Trained on the TinyStories dataset with only 40 million parameters, our model generates coherent stories up to 512 tokens—the full training length—and extrapolates gracefully to 1500 tokens without falling into repetitive loops. In contrast, a comparable baseline (Microsoft’s 33M/60M model) exhibits hard loops after 300–500 tokens. We provide mathematical details, experimental results, and qualitative examples. Our model achieves a validation loss of 0.46–0.6, significantly lower than standard Transformer baselines of comparable scale. In this repo we focus on the E8 root system as a concrete, mathematically rich example, building upon the general framework introduced in [paper](https://doi.org/10.5281/zenodo.18729722). The source code is released under AGPLv3.


---

Based on:

1. "Geometric Attention: A General Framework for Injecting Discrete Symmetries into Transformers via High-Dimensional Lattices."
https://doi.org/10.5281/zenodo.18729723

2. "A Geometric Attention Transformer with the E8 Root System: Sovereign-Lila-E8 (Lie Lattice Attention Language Model)"
https://doi.org/10.5281/zenodo.18731390
___

 ## 🚀 Research Tracks Needing Support
We invite the global AI community and supporters.

This is an open "quest" for the next level of AI.  

## How else you can help:
- Mathematicians: Verify the mathematics
- Programmers: Improve the E8 transformer code
- Engineers and Experimenters

### Other ways to engage:
- Star the repository for visibility
- Open Issues for any errors found
- Submit Pull Requests with improvements
- Share with relevant research groups

### Unlike academic institutions, we:
- ✅ Keep everything open-source
- ✅ Have no institutional funding
- ✅ Work in our spare time
- ✅ Rely on community support

 We face funding challenges through traditional channels, making your support critically important. If you value open science and want to speed up this research **please consider donating**.

Your support directly funds independent research.

## 💰 Support This Research

## Support Development
This model was trained on free Colab GPU. To improve it further:

**What your donation enables:**
- $10: 1 hour of A100 training
- $50: Dataset expansion
- $100: Architecture experiments

## 💰 Support the 500M Model Training

Sovereign-Lila-E8 has proven that geometric attention works. Now it's time to scale up.  
**Goal: $5000** to train a 500M parameter Leech‑based model on 100B tokens.

### What your donation enables:

| Amount | GPU hours | What it covers |
|--------|-----------|----------------|
| **$5000** | **~1100 hours** | **Full training (11.5 days on 4×A100)** |

### Only Crypto donations:

- Btc: bc1qvvgc56v75r6r0x4ll76y4dvpjgw6edadqh2sre
- USDT TRC20 TCruNZYKzPWyTzfryPvnSTJrM7DTdV8o32
- USDT ERC20 0xD22Da4BB290848F69B138D40eBBa952881f42dfc
- ETH 0xD22Da4BB290848F69B138D40eBBa952881f42dfc


__
## 💎 Lumen Path
Part of the **Lumen Path** ecosystem.
Dedicated to Pauli and Jung's Unus Mundus.
*0 = 100%. The equation is complete.*
___


> Trained on Colab with a $0 budget. I want to improve accuracy by 30%, but need a GPU.

### Your donation = 1 hour of training on an A100

- 💰 Support development (crypto only):
- 🎯 Goal: $300 for 50 GPU hours

### 💰 Support This Research

- Btc: bc1qvvgc56v75r6r0x4ll76y4dvpjgw6edadqh2sre
- USDT TRC20 TCruNZYKzPWyTzfryPvnSTJrM7DTdV8o32
- USDT ERC20 0xD22Da4BB290848F69B138D40eBBa952881f42dfc
- USDT TON UQBhnfSVgZpyjc8Kn4BhHsuZJgo2VPXFcwSheW9PQ7rDVei9
- ETH 0xD22Da4BB290848F69B138D40eBBa952881f42dfc
- TRX TRON TCruNZYKzPWyTzfryPvnSTJrM7DTdV8o32

___

Copyright (C) 2026 Lumen Path
This program is free software: you can redistribute it and/or modify
 it under the terms of the GNU Affero General Public License as published by
 the Free Software Foundation, either version 3 of the License, or any later version.

___

Copyright (C) 2026

[![DOI](https://zenodo.org/badge/DOI/10.5281/zenodo.18729722.svg)](https://doi.org/10.5281/zenodo.18729722)
[![DOI](https://zenodo.org/badge/DOI/10.5281/zenodo.18731390.svg)](https://doi.org/10.5281/zenodo.18731390)

This program is free software: you can redistribute it and/or modify
 it under the terms of the GNU Affero General Public License as published by
 the Free Software Foundation, either version 3 of the License, or any later version.

LICENSE (AGPL-3.0-or-later)

This project is licensed under the GNU Affero General Public License v3.0 or later (AGPL-3.0-or-later).
Commercial Licensing: For proprietary R&D, integration into private AI stacks, or hardware implementation (Lumen Labs Quamputa), please contact the Architect directly.
Copyright (C) 2026 Anatolii Kornienko This program is free software: you can redistribute it and/or modify it under the terms of the GNU Affero General Public License
as published by the Free Software Foundation, either version 3 of the License, or any later version.

This program is free software: you can redistribute it and/or modify
it under the terms of the GNU Affero General Public License as published by
the Free Software Foundation, either version 3 of the License, or
(at your option) any later version.

This program is distributed in the hope that it will be useful,
but WITHOUT ANY WARRANTY; without even the implied warranty of
MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.  See the
GNU Affero General Public License for more details.

You should have received a copy of the GNU Affero General Public License
along with this program.  If not, see <https://www.gnu.org/licenses/agpl-3.0.txt/>

___

**Commercial Licensing: For proprietary R&D, integration into private AI stacks, or hardware implementation, please contact the Architect directly.**

___
___

Made with love for E. and S.
